<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/giovanni/MOD-1/notebooks/07_age_prediction_eda_regression_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Age Prediction: data preparation, Exploratory Data Analysis (EDA), and regression with cross-validation (CV) & nested CV

## Data Preparation and Exploratory Data Analysis (EDA)

Import libraries

In [ ]:
# Preparation same as ex 1

## Regression task

### Linear regression using a CV scheme

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold #import tools
from sklearn.svm import SVR
# Setting the seed of the random generator
SEED = 42
# Setting the number of folds
n_folds = 5 #k: determines how many parts dataset will be split into for CV

# Creating the estimator through SVR using as kernel (set of elements in domain mapped to zero element in codomain) the radial basis function rbf, to capture highly non linear relationship
#reg = LinearRegression(); SVR support vector regression, we do not minimize squared error like in regression but ensures that errors fall within this margin
reg = SVR(kernel='rbf', degree=3, gamma='scale', coef0=0.0, tol=0.001, C=1, epsilon=0.1, shrinking=True, cache_size=200, verbose=0, max_iter=- 1) #define the final estimator model
#IN practice what is done in SVR is trying to fit a flexible tube through data, with width equal to parameter epsilon; if points inside this tube no penalty;
#if points outside (SUPPORT VECTORS) the model penalize them through C parameter (complexity);

# Creating the splitter: CROSS VALIDATION splitting strategy, it prevents overfitting --> model trains on 4 folds, test on 1 and then repeat  changing folds --> robust estimate of performance
cv = KFold(n_splits=n_folds, shuffle=True, random_state=SEED) #this will organize the k splits, shuffle active means that before splitting we shuffle the folds to prevent pattern

In [ ]:
# Print the generated splits: prints exact indices of data points assigned to training and testing sets during each of the 5 folds
for train_index, test_index in cv.split(X): #takes as input X and produce folds, then returns the two lists (train and test), this is done at every iteration of the cycle
    print("Train:", train_index, " Test:", test_index)

In [ ]:
score = cross_validate(reg, X=X, y=y, cv=cv, return_train_score=True, scoring = 'neg_mean_absolute_error') #does CV, neg perchè di base cross_validate da come migliore il valore piu alto, vogliamo l'inverso -->
#mean absolute error (MAE) measurees average magnitude of prediction errors; scikit requires scoring metrics to be maximized so convert into a negative number
#K-FOLD cross validation using features(X), target labels (y), SVR model(reg), and splitter defined earlier
print("This is the score object:")
print (score)
print("Average MAE training set:", np.mean(np.abs(score['train_score'])), "years")
print("Average MAE tes set:", np.mean(np.abs(score['test_score'])), "years")

This is the score object:
{'fit_time': array([0.00374651, 0.0036602 , 0.00308919, 0.00242257, 0.002671  ]), 'score_time': array([0.00228858, 0.00211477, 0.00268149, 0.0032196 , 0.00188565]), 'test_score': array([-1.96516495, -1.69834222, -1.47619532, -1.3528025 , -2.07928787]), 'train_score': array([-1.51757213, -1.72181049, -1.69668166, -1.70997242, -1.56123815])}
Average MAE training set: 1.6414549705947792 years
Average MAE tes set: 1.7143585706462816 years


In [ ]:
print(type(score))
print(type(score['train_score']))

### Support Vector Regression with hyperparameter C (Complexity) using a nested CV scheme

In [ ]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# Setting the seed of the random generator
SEED = 42
# Setting the number of folds of both outer and inner k-fold CV --> nested validation setup
outer_n_folds = 5
inner_n_folds = 5

# Setting the possible values of the C hyperparameter: determines the penalty for data points that fall entirely epsilon tube
C = [0.1, 1, 10] #0.1 smoother generalized model, 10 fit training data strictly, risking overfitting

# Creating the splitters: inner loop evaluates value of hyperparameter C (what works best), outer loop provides unbiased evaluation
outer_cv = KFold(n_splits=outer_n_folds, shuffle=True, random_state=SEED)
inner_cv = KFold(n_splits=inner_n_folds, shuffle=True, random_state=SEED)

# Creating the estimator: same as before
reg = SVR(kernel='rbf', degree=3, gamma='scale', coef0=0.0, tol=0.001, C=0.1, epsilon=0.1, shrinking=True, cache_size=200, verbose=0, max_iter=-1)

p_grid = [{'C': C}] # Defining the grid of hyperparameter values
#Grid search is an automated tuning technique: for every step of inner fold it test all values provided in p_grid, grade them using MAE metric and take optimal value
reg_gs = GridSearchCV(reg, param_grid=p_grid, cv=inner_cv, refit='neg_mean_absolute_error', scoring='neg_mean_absolute_error', verbose = 4)
nested_score = cross_validate(reg_gs, X=X, y=y, cv=outer_cv, return_train_score=True, return_estimator=True, scoring = 'neg_mean_absolute_error')
#is used cross_validate function that initiate outer_cv splits; for each split perform Gridsearch perform inner split to find best C value;
#then it trains a final model on the outer split's training data and scores it on the outer split's test data.

#print(np.abs(nested_score['train_score']))
#print(np.abs(nested_score['test_score']))
print("Average MAE train:", np.abs(np.mean(nested_score['train_score'])), "years")
print("Average MAE test:", np.abs(np.mean(nested_score['test_score'])), "years")

In [ ]:
print(type(p_grid))
print(type(p_grid[0]))